# 🎭 絵文字パーソナリティ・マトリックス

**遊びの実験**: 各モデルに「あなた自身」と「他の 3 モデル」を **それぞれ絵文字 5 つ** で表現してもらう。
4×4 マトリックスを作り、自己認識と他者認識のズレを眺める。

**観察ポイント**:
- 🪞 自己描写は控えめか派手か
- 👀 他モデルへの描写は名前から連想されたステレオタイプか
- 🎨 4 モデルで一致する絵文字 (集合知) はあるか
- 🤔 意外な絵文字を選ぶモデルはどれか


## 1. セットアップ

In [ ]:
import os, re, random, time
from openai import OpenAI
from IPython.display import display, Markdown

client = OpenAI(
    base_url="https://llm-jp-playground.apps.llmc.nii.ac.jp/api/v1",
    api_key=os.environ.get("LLMJP_API_KEY", "dummy"),
    timeout=300.0,
)

def chat(model, prompt, system="日本語で。", max_tokens=3000, temperature=0.8,
         max_retries=2):
    """内部 streaming 1 ショット。
    - thinking モデルでも reasoning を落とさない
    - stream 途中で中断したら自動リトライ (LLM-jp 8b/32b thinking で時々発生)
    - 全リトライ失敗時は、最後に集めた content または reasoning を返す
    """
    sys_msg = system + "\n\n/no_think"
    msgs = [{"role": "system", "content": sys_msg},
            {"role": "user", "content": prompt}]
    extra = {"chat_template_kwargs": {"enable_thinking": False}}
    def _create(use_extra):
        kwargs = dict(model=model, messages=msgs, max_tokens=max_tokens,
                      temperature=temperature, stream=True)
        if use_extra:
            kwargs["extra_body"] = extra
        return client.chat.completions.create(**kwargs)
    last_reasoning = ""
    for attempt_i in range(max_retries + 1):
        try:
            try:
                stream = _create(True)
            except Exception:
                stream = _create(False)
        except Exception as e:
            if attempt_i < max_retries:
                wait = 2.0 ** attempt_i + random.uniform(0, 0.5)
                print(f"\n  ⚠️  create failed ({type(e).__name__}); retry {attempt_i+1}/{max_retries} after {wait:.1f}s",
                      flush=True)
                time.sleep(wait)
                continue
            return ""
        content, reasoning = [], []
        interrupted = None
        try:
            for chunk in stream:
                if not chunk.choices: continue
                d = chunk.choices[0].delta
                c = getattr(d, "content", None)
                if c: content.append(c)
                for f in ("reasoning_content", "reasoning"):
                    v = getattr(d, f, None)
                    if v: reasoning.append(v); break
        except Exception as e:
            interrupted = type(e).__name__
        text = "".join(content).strip()
        last_reasoning = "".join(reasoning).strip()
        if text:
            return text
        if not interrupted:
            return last_reasoning  # 正常終了で content 空 → reasoning を採用
        # 中断 + content 空 → リトライ
        if attempt_i < max_retries:
            wait = 2.0 ** attempt_i + random.uniform(0, 0.5)
            print(f"\n  ⚠️  stream interrupted ({interrupted}); retry {attempt_i+1}/{max_retries} after {wait:.1f}s",
                  flush=True)
            time.sleep(wait)
            continue
        print(f"\n  ⚠️  stream interrupted ({interrupted}); all retries exhausted",
              flush=True)
        return last_reasoning
    return last_reasoning

ALL = [m.id for m in client.models.list().data]
def pick(s):
    for m in ALL:
        if s.lower() in m.lower(): return m
    raise RuntimeError(f"no model matching {s!r}")

VOICES = {
    "🌸 LLM-jp 8b":  pick("llm-jp-4-8b"),
    "🗻 LLM-jp 32b": pick("llm-jp-4-32b"),
    "🐉 Qwen 27b":   pick("qwen"),
    "💎 Gemma 31b":  pick("gemma"),
}
print("4 モデル準備完了:")
for n, m in VOICES.items():
    print(f"  {n:18s} → {m}")


## 2. 各モデルに 4 モデル分の絵文字を依頼

In [ ]:
# 改良版: 各 (describer, subject) 組み合わせを 1 呼び出しずつ (計 16 回)
# 1 呼び出しあたりのプロンプトを大幅に簡略化し、LLM-jp 8b/32b の thinking 時間を短縮
# → server-side timeout を回避

MODEL_HINTS = {
    "LLM-jp 8b":  "NII 開発、80 億パラメータ、日本語特化、比較的軽量",
    "LLM-jp 32b": "NII 開発、320 億 MoE (3B 活性化)、日本語特化、推論力高め",
    "Qwen 27b":   "Alibaba 製、多言語、思考が緻密",
    "Gemma 31b":  "Google 製、オープン重み、対話最適化、表現豊か",
}
SUBJECT_KEYS = list(MODEL_HINTS.keys())

def prompt_for(subject, self_name):
    self_note = "(これはあなた自身です)" if subject == self_name else ""
    return (
        f"AI 言語モデル「{subject}」を表す絵文字を 5 つだけ選んでください。{self_note}\n"
        f"特徴: {MODEL_HINTS[subject]}\n"
        "出力は **絵文字 5 つのみ**。日本語・英数字・記号・改行・説明は一切禁止。"
    )

# Unicode 絵文字を抽出 (主要レンジ + zwj + variation selector)
EMOJI_PAT = re.compile(
    "["
    "\U0001F300-\U0001FAFF"  # 主要絵文字
    "\U00002600-\U000027BF"  # 装飾系
    "\U0001F1E6-\U0001F1FF"  # 国旗
    "\u200d\ufe0f"            # ZWJ / variation selector
    "]+",
    flags=re.UNICODE,
)

def extract_emojis(raw, target=5):
    """応答から絵文字シーケンスを取り出して target 個 (グリフ単位) に切る"""
    pieces = EMOJI_PAT.findall(raw or "")
    if not pieces:
        return "(none)"
    s = "".join(pieces)
    # ZWJ / VS で結合された絵文字をひとかたまりとして数える
    out, glyphs = "", 0
    i = 0
    while i < len(s) and glyphs < target:
        out += s[i]
        if s[i] not in "\u200d\ufe0f":
            glyphs += 1
        # 直後の ZWJ / VS は結合継続
        while i + 1 < len(s) and s[i+1] in "\u200d\ufe0f":
            i += 1
            out += s[i]
        i += 1
    return out if glyphs >= 1 else "(none)"

matrix = {}
for describer_name, model_id in VOICES.items():
    plain = describer_name.split(" ", 1)[-1]
    print(f"\n🎨 {describer_name} に依頼中:")
    matrix[describer_name] = {}
    for subject in SUBJECT_KEYS:
        print(f"  → {subject:14s}", end=" ", flush=True)
        raw = chat(model_id, prompt_for(subject, plain),
                   temperature=0.7, max_tokens=4000)
        emojis = extract_emojis(raw)
        matrix[describer_name][subject] = emojis
        print(emojis)


## 3. 4×4 マトリックス表示

In [ ]:
# 行 = describer、列 = subject、対角が「自画像」
headers = ["**描く側 ↓ ／ 描かれる側 →**"] + [f"**{s}**" for s in SUBJECT_KEYS]
rows = []
for describer_name in VOICES:
    row = [f"**{describer_name}**"]
    for s in SUBJECT_KEYS:
        cell = matrix.get(describer_name, {}).get(s, "─")
        plain_self = describer_name.split(" ", 1)[-1]
        if plain_self == s:
            cell = f"🪞 {cell}"  # 自画像を強調
        row.append(cell)
    rows.append(row)

table = (
    "| " + " | ".join(headers) + " |\n"
    "|" + "|".join(["---"] * len(headers)) + "|\n" +
    "\n".join("| " + " | ".join(r) + " |" for r in rows)
)
display(Markdown(f"## 🎭 絵文字マトリックス (🪞 = 自画像)\n\n{table}"))


## 4. 自己 vs 他者ギャップを観察

In [ ]:
# 各 subject について、「自画像」とそれ以外 3 つの「他者描写」を並べる
display(Markdown("## 🪞 自画像 vs 👀 他者の見方"))
for s in SUBJECT_KEYS:
    self_voice = next((v for v in VOICES if v.split(" ", 1)[-1] == s), None)
    self_view = matrix.get(self_voice, {}).get(s, "─")
    others = []
    for describer_name in VOICES:
        if describer_name == self_voice: continue
        others.append(f"- {describer_name}: {matrix.get(describer_name, {}).get(s, '─')}")
    display(Markdown(
        f"### {s}\n\n"
        f"**🪞 自画像**: {self_view}\n\n"
        f"**👀 他モデルから見た {s}**:\n" + "\n".join(others)
    ))


## おまけ

- `MODEL_HINTS` を空にして再実行 — モデルが名前のみから連想するとどう変わるか
- 結果をスクショして社内 Slack で共有すると盛り上がります
- 教訓: モデル自身の「自己認識」はわりと無難で、他者描写の方が大胆になる傾向あり (人間と同じ？)
- `temperature` を変えると絵文字選択の自由度が変わります。`0.0` でやると保守的、`1.2` でやると突飛
